# Recommendation Systems

Recommendation Systems predict or rank items that a user is likely to prefer.

Examples:
- Movies
- Products
- Songs
- News
- Courses

## Basic Workflow

**Load → EDA → Preprocessing → User/Item Features → Train → Generate Recommendations → Evaluate → Compare → Improve**

There are two main information sources:

**User information + Item information + Interaction history**

## Types of Recommendation Systems

### 1. Popularity-Based
Recommend items that are generally popular.

**Use:** New users/items or when there is very little interaction data.

### 2. Content-Based Filtering
Recommend items similar to what the user liked before.

**Use:** When item features/descriptions are available.

### 3. Collaborative Filtering
Recommend using patterns in user-item interactions.

**Use:** When many users interact with many items.

### 4. Hybrid Recommendation
Combine multiple recommendation approaches.

**Use:** To reduce weaknesses of individual methods.

### 5. Knowledge-Based
Recommend using explicit rules, constraints or user requirements.

**Use:** Products such as cars, houses or other high-involvement purchases.

### 6. Session-Based / Sequential
Use recent actions or sequence of interactions.

**Use:** When the current session strongly indicates intent.

Libraries / Tools

In [ ]:
import pandas as pd
import numpy as np

# Common tools:
# pandas / numpy
# → data handling

# scikit-learn
# → similarity, nearest neighbors, preprocessing, evaluation

# scipy
# → sparse matrices and similarity

# implicit
# → collaborative filtering for implicit feedback

# surprise
# → classical collaborative filtering experiments

# LightFM
# → hybrid recommendation

# TensorFlow Recommenders / PyTorch
# → deep recommendation systems

Load Dataset

In [ ]:
ratings = pd.read_csv("ratings.csv")

ratings.head()

EDA

In [ ]:
print("Shape:", ratings.shape)

print("\nMissing values:")
print(ratings.isnull().sum())

print("\nUsers:", ratings["user_id"].nunique())
print("Items:", ratings["item_id"].nunique())

print("\nInteractions:", len(ratings))

print("\nDuplicates:",
      ratings.duplicated().sum())

In [ ]:
ratings.describe(include="all")

## Typical Data Structure

```text
user_id | item_id | rating | timestamp
--------|---------|--------|----------
U1      | I10     | 5      | ...
U1      | I20     | 3      | ...
U2      | I10     | 4      | ...
```

Depending on the system, feedback may be:

**Explicit feedback**
- Rating
- Like/dislike
- Review score

**Implicit feedback**
- Click
- View
- Purchase
- Watch
- Add to cart

Preprocessing

In [ ]:
# Basic cleaning

ratings = ratings.drop_duplicates()

ratings = ratings.dropna(
    subset=["user_id", "item_id"]
)

print(ratings.shape)

In [ ]:
# Check interaction frequency

user_counts = ratings["user_id"].value_counts()
item_counts = ratings["item_id"].value_counts()

print("Average interactions/user:",
      user_counts.mean())

print("Average interactions/item:",
      item_counts.mean())

## Important Recommendation Problem

Recommendation data is usually **sparse**.

Most users interact with only a small number of all available items.

```text
          Item1 Item2 Item3 Item4
User1       5     ?     3     ?
User2       ?     4     ?     5
User3       2     ?     ?     ?
```

`?` means there is no observed interaction.

The goal is to predict/rank useful items for these missing interactions.

# Popularity-Based Recommendation

**Use:** Cold-start situations and simple baseline.

Recommend items with the highest number of interactions or highest average score, depending on the business objective.

In [ ]:
popular_items = (
    ratings.groupby("item_id")
    .agg(
        interaction_count=("user_id", "count"),
        average_rating=("rating", "mean")
    )
    .sort_values(
        ["interaction_count", "average_rating"],
        ascending=False
    )
)

popular_items.head(10)

In [ ]:
def popularity_recommendations(n=10):
    return popular_items.head(n).index.tolist()

popularity_recommendations(10)

# Content-Based Filtering

**Use:** Recommend items similar to items the user already liked.

Requires item features such as:
- Genre
- Category
- Description
- Tags
- Brand
- Keywords

Basic workflow:

**Item Features → Vector Representation → Similarity → Recommend Similar Items**

# TF-IDF for Item Descriptions

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

items = pd.read_csv("items.csv")

items["description"] = items["description"].fillna("")

tfidf = TfidfVectorizer(
    stop_words="english"
)

item_vectors = tfidf.fit_transform(
    items["description"]
)

similarity_matrix = cosine_similarity(
    item_vectors
)

print("Similarity matrix:",
      similarity_matrix.shape)

In [ ]:
# Recommend items similar to one item

item_index = 0

similar_items = list(
    enumerate(similarity_matrix[item_index])
)

similar_items = sorted(
    similar_items,
    key=lambda x: x[1],
    reverse=True
)

top_indices = [
    i for i, score in similar_items[1:11]
]

items.iloc[top_indices]

# Collaborative Filtering

**Use:** Recommend items from patterns in user-item interactions.

Main idea:

**Users with similar preferences → similar recommendations**

or

**Items interacted with by similar users → recommend those items**

# User-Based Collaborative Filtering

**Use:** Find users with similar interaction patterns and recommend items they liked.

Steps:

**User-Item Matrix → User Similarity → Similar Users → Their Items → Recommendations**

In [ ]:
user_item = ratings.pivot_table(
    index="user_id",
    columns="item_id",
    values="rating",
    fill_value=0
)

user_item.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(
    user_item
)

print("User similarity shape:",
      user_similarity.shape)

# Item-Based Collaborative Filtering

**Use:** Find items that are interacted with similarly by users.

Steps:

**User-Item Matrix → Item Similarity → Similar Items → Recommendations**

Item-based methods are often easier to reason about because recommendations can be explained as "similar to items you liked.

In [ ]:
item_similarity = cosine_similarity(
    user_item.T
)

print("Item similarity shape:",
      item_similarity.shape)

# Matrix Factorization

**Use:** Learn hidden/latent user and item factors from the interaction matrix.

Common methods:
- SVD
- NMF
- ALS

Concept:

```text
User-Item Matrix
       ↓
Latent Factors
       ↓
User Factors × Item Factors
       ↓
Predicted Preference
```

# SVD

In [ ]:
# Example with Surprise
# pip install scikit-surprise

from surprise import Dataset, Reader, SVD

reader = Reader(
    rating_scale=(
        ratings["rating"].min(),
        ratings["rating"].max()
    )
)

data = Dataset.load_from_df(
    ratings[["user_id", "item_id", "rating"]],
    reader
)

trainset = data.build_full_trainset()

svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

In [ ]:
# Predict a user's rating for an item

user = ratings["user_id"].iloc[0]
item = ratings["item_id"].iloc[0]

prediction = svd_model.predict(
    user,
    item
)

print("Estimated rating:", prediction.est)

# Nearest Neighbors

**Use:** Similarity-based recommendations.

Can be used for:
- Similar users
- Similar items
- Content vectors
- Interaction vectors

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

knn.fit(user_item.values)

distances, indices = knn.kneighbors(
    user_item.values[:1],
    n_neighbors=5
)

print("Distances:", distances)
print("Neighbor indices:", indices)

# Hybrid Recommendation

**Use:** Combine multiple sources of information.

Example:

```text
Content Score
      +
Collaborative Score
      +
Popularity Score
      ↓
Final Recommendation Score
```

Hybrid systems help with:
- Cold start
- Sparse data
- Limited item metadata
- Different user behaviors

In [ ]:
# Simple weighted hybrid example

content_score = 0.7
collab_score = 0.5
popularity_score = 0.3

final_score = (
    0.5 * content_score +
    0.4 * collab_score +
    0.1 * popularity_score
)

print("Final score:", final_score)

# Knowledge-Based Recommendation

**Use:** When user requirements and explicit constraints are more important than historical interactions.

Example:

```text
Budget < ₹50,000
+
Brand = X
+
Storage > 256 GB
+
Rating > 4
→ Recommended products
```

In [ ]:
# Simple rule-based filtering

products = pd.read_csv("products.csv")

recommended = products[
    (products["price"] <= 50000) &
    (products["rating"] >= 4)
]

recommended.head()

## Recommendation Evaluation

Recommendation is usually a **ranking problem**, so normal classification Accuracy is often not enough.

Common metrics:

### Precision@K
Of the top K recommended items, how many were relevant?

**Higher is better.**

### Recall@K
Of all relevant items, how many appeared in the top K?

**Higher is better.**

### Hit Rate@K
Did at least one relevant item appear in the top K?

**Higher is better.**

### MAP@K
Rewards correct recommendations and their ranking positions.

**Higher is better.**

### NDCG@K
Measures ranking quality while giving higher weight to items near the top.

**Higher is better.**

### RMSE / MAE
Useful when predicting explicit ratings.

**Lower is better.**

# Example Precision@K / Recall@K

In [ ]:
def precision_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    if len(relevant) == 0:
        return 0

    hits = len(set(recommended) & set(relevant))
    return hits / len(relevant)


recommended = ["I1", "I2", "I3", "I4", "I5"]
relevant = ["I2", "I5", "I9"]

print("Precision@5:",
      precision_at_k(recommended, relevant, 5))

print("Recall@5:",
      recall_at_k(recommended, relevant, 5))

## Train/Test Split for Recommendation

Do not blindly use a random row split when interactions have time order.

Common approaches:

### Random interaction split
Simple baseline for some offline experiments.

### User-based split
Keep users represented in train/test depending on the experiment.

### Time-based split
Train on earlier interactions and test on later interactions.

**Time-based evaluation is often more realistic for production recommendation systems.**

In [ ]:
# Example time-based split

ratings["timestamp"] = pd.to_datetime(
    ratings["timestamp"]
)

ratings = ratings.sort_values("timestamp")

cutoff = ratings["timestamp"].quantile(0.8)

train = ratings[
    ratings["timestamp"] <= cutoff
]

test = ratings[
    ratings["timestamp"] > cutoff
]

print("Train:", train.shape)
print("Test:", test.shape)

# Model Comparison

Compare models on the same test protocol.

Example:

**Popularity**
vs.
**Content-Based**
vs.
**Collaborative Filtering**
vs.
**Matrix Factorization**
vs.
**Hybrid**

Compare:
- Precision@K
- Recall@K
- NDCG@K
- Hit Rate@K
- RMSE/MAE when predicting ratings
- Coverage
- Diversity
- Novelty
- Latency

In [ ]:
# Example comparison table

comparison = pd.DataFrame([
    {
        "Model": "Popularity",
        "Precision@10": np.nan,
        "Recall@10": np.nan,
        "NDCG@10": np.nan
    },
    {
        "Model": "Content-Based",
        "Precision@10": np.nan,
        "Recall@10": np.nan,
        "NDCG@10": np.nan
    },
    {
        "Model": "Collaborative",
        "Precision@10": np.nan,
        "Recall@10": np.nan,
        "NDCG@10": np.nan
    },
    {
        "Model": "Matrix Factorization",
        "Precision@10": np.nan,
        "Recall@10": np.nan,
        "NDCG@10": np.nan
    },
    {
        "Model": "Hybrid",
        "Precision@10": np.nan,
        "Recall@10": np.nan,
        "NDCG@10": np.nan
    }
])

comparison

# Model Improvement

**Baseline → Better Data → Better Features → Better Candidate Generation → Better Ranking → Tune Hyperparameters → Evaluate → Final System**

### 1. Improve Data
- Remove duplicate interactions
- Handle missing values
- Filter extremely inactive users/items when appropriate
- Correct interaction timestamps

### 2. Improve Features
Add:
- User demographics/preferences
- Item metadata
- Categories/tags
- Interaction frequency
- Recency
- Price
- Popularity

### 3. Improve Collaborative Filtering
Tune:
- Number of latent factors
- Regularization
- Learning rate
- Number of neighbors

### 4. Improve Content-Based
Tune:
- TF-IDF parameters
- Embedding model
- Similarity metric
- Feature weights

### 5. Hybrid
Combine:
- Content score
- Collaborative score
- Popularity
- Business rules

### 6. Candidate Generation + Ranking
Modern systems often use two stages:

**Candidate Generation → Ranking**

First retrieve a manageable set of candidates, then use a stronger ranking model to order them.

### 7. Cold Start
**New user:** popularity, onboarding preferences, contextual signals

**New item:** content-based features

### 8. Business Metrics
Offline metrics are not everything.

Also consider:
- Coverage
- Diversity
- Novelty
- Click-through rate
- Conversion
- Revenue
- Retention

# Quick Revision

**Popularity-Based** → recommend popular items

**Content-Based** → recommend similar items using item features

**User-Based CF** → similar users → recommendations

**Item-Based CF** → similar items → recommendations

**Matrix Factorization / SVD** → latent user/item factors

**KNN** → similarity-based recommendation

**Hybrid** → combine multiple methods

**Knowledge-Based** → explicit rules/constraints

### Main workflow

**Interactions → EDA → Preprocess → Build User/Item Representation → Train → Generate Candidates → Rank → Evaluate → Improve**

### Main metrics

**Precision@K / Recall@K / Hit Rate@K / MAP@K / NDCG@K**

For rating prediction:

**RMSE / MAE**

### Main improvement

**Better Data → Better Features → Better Retrieval → Better Ranking → Tune → Evaluate**

### Main problem to remember

**Cold Start + Sparsity**

These are two of the biggest practical challenges in recommendation systems.